# MGGP — Nguyen-10 & Double Well Potential (2D)
**Multi-population MGGP** on two 2D problems `(x1, x2)`.

Showcases library features:
- `regression_degree=2` — polynomial gene features
- `RankSelection` — graded selection pressure
- `missing_vars_penalty` — ensures all variables are used
- `schedule` — adaptive parameters per generation
- `EarlyStopping` — automatic stop on convergence

Figures produced:
- `fig04_mggp_obs_pred.png` — Observed × Predicted
- `fig04_mggp_fitness_decay.png` — Fitness decay
- `fig04_mggp_surfaces.png` — 3D Surfaces: true function vs MGGP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
from sklearn.metrics import r2_score
import os, math
import pandas as pd

FIG_DIR = Path("figures")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_COLOR = '#1f77b4'
VAL_COLOR   = '#ff7f0e'
TEST_COLOR  = '#2ca02c'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.linestyle': '-', 'grid.alpha': 0.4,
    'grid.color': '#cccccc', 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 13, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

def obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title,
                xlabel="Observed", ylabel="Predicted"):
    h1 = ax.scatter(y_tr, yp_tr, c=TRAIN_COLOR, marker='o', s=40, alpha=0.75,
                    label=f'Train  (R²={r2_score(y_tr, yp_tr):.6f})')
    h2 = ax.scatter(y_v,  yp_v,  c=VAL_COLOR,   marker='s', s=40, alpha=0.75,
                    label=f'Val    (R²={r2_score(y_v,  yp_v):.6f})')
    h3 = ax.scatter(y_te, yp_te, c=TEST_COLOR,  marker='^', s=40, alpha=0.75,
                    label=f'Test   (R²={r2_score(y_te, yp_te):.6f})')
    all_y = np.concatenate([y_tr, y_v, y_te])
    lo, hi = all_y.min(), all_y.max()
    ax.plot([lo, hi], [lo, hi], color='black', lw=1.5)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    return h1, h2, h3

from symgene.callbacks import Callback

class HistoryRecorder(Callback):
    def __init__(self):
        self.records = []
    def on_generation_end(self, gen, logs=None):
        if logs:
            self.records.append(dict(logs))
        return None

print("Style loaded.")

def _snap(ax, log_y=False):
    ax.figure.canvas.draw()
    ax.tick_params(top=True, right=True, which='both', direction='in')
    lo, hi = ax.get_xlim()
    xt = sorted(t for t in ax.get_xticks() if lo - 1e-9 <= t <= hi + 1e-9)
    if len(xt) >= 2:
        ax.set_xlim(xt[0], xt[-1])
    if log_y:
        lo, hi = ax.get_ylim()
        if lo > 0 and hi > 0:
            lo_dec = 10 ** math.floor(math.log10(lo))
            hi_log = math.log10(hi)
            frac   = hi_log - math.floor(hi_log)
            hi_dec = 10 ** (math.floor(hi_log) if frac < 0.02 else math.ceil(hi_log))
            if lo_dec < hi_dec:
                ax.set_ylim(lo_dec, hi_dec)
    else:
        lo, hi = ax.get_ylim()
        yt = sorted(t for t in ax.get_yticks() if lo - 1e-9 <= t <= hi + 1e-9)
        if len(yt) >= 2:
            ax.set_ylim(yt[0], yt[-1])

def _fix_cbar(cb, cp):
    _lo, _hi = cp.get_clim()
    _t = [x for x in cb.get_ticks() if _lo < x < _hi]
    cb.set_ticks([_lo] + _t + [_hi])


## Data Generation
Latin Hypercube Sampling 2D, 70/15/15 split.

In [ ]:
DOMAIN = (-1.5, 1.5)

# ─── Analytical function definitions ─────────────────────────────────────────
def nguyen10(x1, x2):
    return 2.0 * np.sin(x1) * np.cos(x2)

def doublewell(x1, x2):
    return x1**4 + x2**4 - 2.0*(x1**2 + x2**2)

# ─── Latin Hypercube Sampling 2D ─────────────────────────────────────────────
def lhs2d(rng, lo, hi, n):
    X = np.empty((n, 2))
    for j in range(2):
        perm = rng.permutation(n)
        X[:, j] = lo + (perm + rng.uniform(0, 1, n)) / n * (hi - lo)
    return X

rng   = np.random.default_rng(42)
N     = 400
X_all = lhs2d(rng, DOMAIN[0], DOMAIN[1], N)
idx   = rng.permutation(N)

y_n_all = nguyen10(X_all[:, 0], X_all[:, 1])
y_d_all = doublewell(X_all[:, 0], X_all[:, 1])

n_train, n_val = 280, 60   # 70 / 15 / 15
i_tr = idx[:n_train]
i_v  = idx[n_train:n_train + n_val]
i_te = idx[n_train + n_val:]

X_train, X_val, X_test = X_all[i_tr], X_all[i_v], X_all[i_te]
y_n_train, y_n_val, y_n_test = y_n_all[i_tr], y_n_all[i_v], y_n_all[i_te]
y_d_train, y_d_val, y_d_test = y_d_all[i_tr], y_d_all[i_v], y_d_all[i_te]

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Domain: x1, x2 in [{DOMAIN[0]}, {DOMAIN[1]}]")
print(f"Nguyen-10  range: [{y_n_all.min():.3f}, {y_n_all.max():.3f}]")
print(f"DoubleWell range: [{y_d_all.min():.3f}, {y_d_all.max():.3f}]")

## PrimitiveSet & Callbacks
A single shared PrimitiveSet — both populations receive x1 and x2.
Each population uses **distinct selectors and penalties** to demonstrate the API.

In [ ]:
from symgene import PrimitiveSet, Population, SymGeneEvolver
from symgene.primitives import STANDARD
from symgene.fitness import FitnessEvaluator
from symgene.metrics.regression import mse
from symgene.metrics.complexity import complexity_penalty, missing_vars_penalty
from symgene.metrics import rmse, nrmse
from symgene.selection import TournamentSelection
from symgene.selection.rank import RankSelection
from symgene.callbacks import EarlyStopping, GenerationLogger

history_cb = HistoryRecorder()

# Shared PrimitiveSet — both populations receive the same inputs x1, x2
pset = PrimitiveSet(n_inputs=2, feature_names=["x1", "x2"])
pset.add_from_catalog(STANDARD)
pset.set_squash(lim=10, alpha=0.08, scale=2.5)   # numerical explosion control

print(f"Primitives loaded: {len(pset.primitives)}")
print(f"Feature names: {pset.feature_names}")

## Populations

| Parameter | Nguyen-10 | Double Well |
|---|---|---|
| Selection | `RankSelection(1.8)` | `TournamentSelection(7)` |
| `regression_degree` | **2** (polynomial gene features) | 1 |
| Penalties | `complexity` | `complexity` + `missing_vars` |
| `mutation_weights` | `[0.3, 1.8, 0.9]` | `[0.2, 2.0, 0.8]` |
| `schedule` | `mutpb` decaying | — |

In [ ]:
# ═══ Nguyen-10: 2*sin(x1)*cos(x2) ══════════════════════════════════════════
# regression_degree=2 : creates polynomial gene features (e.g. sin(x1)*cos(x2))
# RankSelection        : graded selection pressure — favours multimodal functions
# schedule             : mutation rate decays as convergence progresses
pop_nguyen = Population(
    name="nguyen10", pset=pset,
    n_genes=2, n_genes_max=6,
    pop_size=80,
    elite_ratio=0.05,
    tree_min=2, tree_max=25, tree_init_max=3, height_max=6,
    cxpb=0.85, cxpb_low=0.55,
    mutpb=0.35, mutpb_low=0.20,
    mutation_weights=[0.3, 1.8, 0.9],
    combiner="ridge",
    ridge_alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    regression_degree=2,                               # <<< polynomial gene features
    fitness=FitnessEvaluator(metric=mse, penalties=[
        complexity_penalty(lambda_=3e-4),
    ]),
    selection=RankSelection(pressure=1.8),             # <<< rank selection
    schedule={"mutpb": {0: 0.40, 60: 0.30, 100: 0.20}},  # <<< adaptive schedule
)

# ═══ Double Well: x1^4+x2^4-2*(x1^2+x2^2) ════════════════════════════════════
# missing_vars_penalty  : penalises solutions that ignore x1 or x2 (both variables required)
# mutation_weights      : [0.2, 2.0, 0.8] — strongly favours gene addition
# TournamentSelection(7): large tournament = high selection pressure (fast convergence)
pop_dwell = Population(
    name="doublewell", pset=pset,
    n_genes=2, n_genes_max=10,
    pop_size=60,
    elite_ratio=0.03,
    tree_min=2, tree_max=35, tree_init_max=2, height_max=7,
    cxpb=0.90, cxpb_low=0.40,
    mutpb=0.35, mutpb_low=0.25,
    mutation_weights=[0.2, 2.0, 0.8],                 # <<< favors gene addition
    combiner="ridge",
    ridge_alphas=[0.01, 0.1, 1.0, 10.0, 100.0],
    regression_degree=1,
    fitness=FitnessEvaluator(metric=mse, penalties=[
        complexity_penalty(lambda_=1e-4),
        missing_vars_penalty(required={"x1", "x2"}, beta=0.05),  # <<< both vars required
    ]),
    selection=TournamentSelection(size=7),             # <<< high selection pressure
)

print("Populations configured:")
print(f"  nguyen10   — pop={pop_nguyen.pop_size}  genes={pop_nguyen.n_genes}->{pop_nguyen.n_genes_max}"
      f"  degree={pop_nguyen.regression_degree}  sel={pop_nguyen.selection.__class__.__name__}")
print(f"  doublewell — pop={pop_dwell.pop_size}   genes={pop_dwell.n_genes}->{pop_dwell.n_genes_max}"
      f"  degree={pop_dwell.regression_degree}  sel={pop_dwell.selection.__class__.__name__}")

In [ ]:
evolver = SymGeneEvolver(
    populations=[pop_nguyen, pop_dwell],
    n_gen=150,
    cross_population=False,   # distinct problems — no gene exchange between populations
    seed=42,
    callbacks=[
        history_cb,
        GenerationLogger(every=30),
        EarlyStopping(                     # <<< stops automatically on convergence
            monitor="doublewell_train_mse",
            patience=50, mode="min",
        ),
    ],
    verbose=0,
)

results = evolver.fit(
    X_train,
    {"nguyen10": y_n_train, "doublewell": y_d_train},
    X_val=X_val,
    y_val={"nguyen10": y_n_val, "doublewell": y_d_val},
)
print("Training complete.")

## Figure 1 — Observed × Predicted

In [ ]:
# 1. Initialize figure with two side-by-side axes
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plt.subplots_adjust(wspace=0.35, bottom=0.12)

# Explicitly map X and y for each experiment in the config tuple
configs = [
    (
        "nguyen10",
        r"Nguyen-10  —  $2\cdot\sin(x_1)\cdot\cos(x_2)$",
        X_train,
        X_val,
        X_test,
        y_n_train,
        y_n_val,
        y_n_test,
    ),
    (
        "doublewell",
        r"Double Well  —  $x_1^4 + x_2^4 - 2(x_1^2 + x_2^2)$",
        X_train,
        X_val,
        X_test,
        y_d_train,
        y_d_val,
        y_d_test,
    ),
]

for ax, (pop_name, title, X_tr, X_v, X_te, y_tr, y_v, y_te) in zip(
    axes, configs
):
    res = results[pop_name]

    # Use the correct X matrix for each benchmark
    yp_tr = res.predict(X_tr)
    yp_v = res.predict(X_v)
    yp_te = res.predict(X_te)

    # Draw base data on current axis
    obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title)

    # Rewrite legend labels with high precision (8 decimal places)
    r2_tr = r2_score(y_tr, yp_tr)
    r2_v = r2_score(y_v, yp_v)
    r2_te = r2_score(y_te, yp_te)

    # Get plot handles and apply high-precision labels
    handles, _ = ax.get_legend_handles_labels()
    labels = [
        f"Train  (R²={r2_tr:.8f})",
        f"Val    (R²={r2_v:.8f})",
        f"Test   (R²={r2_te:.8f})",
    ]

    # Remove extra borders from scatter markers
    for collection in ax.collections:
        collection.set_linewidth(0)
        collection.set_edgecolor("none")

    for line in ax.lines:
        if line.get_marker() != "None" and line.get_marker() != "":
            line.set_linewidth(0)
            line.set_markeredgewidth(0)

    # Apply _snap styling if defined
    if "_snap" in globals():
        _snap(ax)

    # Add dashed grid
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

    # Ensure solid black formatting on axes and ticks
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # Draw custom legend with expanded R² values on both panels
    leg = ax.legend(
        handles,
        labels,
        loc="upper left",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
        handlelength=1.2,
        handletextpad=0.5,
    )

    # Remove internal lines from legend icons
    for handle in leg.legend_handles:
        if hasattr(handle, "set_linewidth"):
            handle.set_linewidth(0)

# Save high-resolution output
output_path = os.path.join(FIG_DIR, "fig04_mggp_obs_pred.png")
plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print(f"Saved: {output_path}")

## Figure 2 — Fitness decay

In [ ]:
df = pd.DataFrame(history_cb.records)
print("Available columns:", df.columns.tolist())
print(f"Generations recorded: {len(df)}")

pop_configs = [
    ("nguyen10", r"Nguyen-10  —  $2\cdot\sin(x_1)\cdot\cos(x_2)$"),
    ("doublewell", r"Double Well  —  $x_1^4 + x_2^4 - 2(x_1^2 + x_2^2)$"),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plt.subplots_adjust(wspace=0.45, bottom=0.15)

for ax, (pop_name, func_label) in zip(axes, pop_configs):
    col_tr = f"{pop_name}_train_mse"
    if col_tr in df.columns:
        ax.semilogy(
            df["gen"],
            df[col_tr],
            color=TRAIN_COLOR,
            lw=2.5,
            zorder=5,
            label="Best individual",
        )

    ax.set_xlabel("Generation", color="black")
    ax.set_ylabel("Fitness — penalised MSE (log scale)", color="black")
    ax.set_title(func_label, color="black")
    ax.legend(
        loc="upper right", frameon=True, edgecolor="black", fancybox=False
    )

    # 1. Apply _snap styling first
    if "_snap" in globals():
        _snap(ax, log_y=True)

    # 2. Add grid (major and minor lines for log scale)
    ax.grid(
        True,
        which="both",
        linestyle="--",
        linewidth=0.5,
        alpha=0.6,
        color="gray",
    )

    # 3. Ensure visible black borders and tick marks
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

plt.savefig(
    os.path.join(FIG_DIR, "fig04_mggp_fitness_decay.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print("Saved: fig04_mggp_fitness_decay.png")

## Figure 3 — 3D Surfaces: True Function vs MGGP Prediction (overlaid)
Continuous surface: true analytical function (cmap `viridis`).
Red dots: MGGP predictions evaluated on a regular grid — overlaid on the true surface.
Point alignment with the surface demonstrates the fit quality.

In [ ]:
# Dense grid for the true surface
x1g = np.linspace(DOMAIN[0], DOMAIN[1], 80)
x2g = np.linspace(DOMAIN[0], DOMAIN[1], 80)
X1, X2 = np.meshgrid(x1g, x2g)

# Sparse grid for MGGP prediction points (25×25 = 625 points)
x1s = np.linspace(DOMAIN[0], DOMAIN[1], 25)
x2s = np.linspace(DOMAIN[0], DOMAIN[1], 25)
X1s, X2s = np.meshgrid(x1s, x2s)
X_scatter = np.column_stack([X1s.ravel(), X2s.ravel()])

configs_3d = [
    (
        "nguyen10",
        r"Nguyen-10:  $2\sin(x_1)\cos(x_2)$",
        nguyen10(X1, X2),
    ),
    (
        "doublewell",
        r"Double Well:  $x_1^4+x_2^4-2(x_1^2+x_2^2)$",
        doublewell(X1, X2),
    ),
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6),
    subplot_kw={"projection": "3d"},
    gridspec_kw={"wspace": -0.15},
)

plt.suptitle(
    "True Function (surface) vs MGGP Prediction (scatter)",
    fontsize=13,
    y=0.98,
)

ELEV, AZIM = 28, -55

for ax, (pop_name, title, Z_true) in zip(axes, configs_3d):
    Z_pred = results[pop_name].predict(X_scatter).reshape(X1s.shape)

    X_te = X_test
    y_te = y_n_test if pop_name == "nguyen10" else y_d_test

    r2_te = r2_score(y_te, results[pop_name].predict(X_te))

    ax.computed_zorder = False

    # ── 1. True surface ──────────────────────────────────────────────────────
    ax.plot_surface(
        X1,
        X2,
        Z_true,
        cmap="viridis",
        alpha=0.75,
        linewidth=0,
        antialiased=True,
    )

    # ── 2. MGGP predictions ──────────────────────────────────────────────────
    ax.scatter(
        X1s.ravel(),
        X2s.ravel(),
        Z_pred.ravel(),
        c="red",
        s=2,
        alpha=1.0,
        depthshade=False,
        label=f"MGGP  R²={r2_te:.8f}",
    )

    ax.set_title(title + f"\nR² (test) = {r2_te:.8f}", fontsize=11, pad=8)

    # X and Y axis labels
    ax.set_xlabel("$x_1$", color="black", labelpad=3)
    ax.set_ylabel("$x_2$", color="black", labelpad=3)

    # Z label: locked at 90°
    ax.zaxis.set_rotate_label(False)
    ax.set_zlabel(
        "$f(x_1,x_2)$",
        color="black",
        fontsize=10,
        labelpad=2,
        rotation=90,
        ha="right",
        va="bottom",
    )

    # ── Fix Z axis limits and ticks ──────────────────────────────────────────
    if pop_name == "nguyen10":
        # Nguyen-10 oscillates between -2.0 and +2.0
        ax.set_zlim(-2.0, 2.0)
        ax.set_zticks([-2.0, -1.0, 0.0, 1.0, 2.0])
    else:
        # Double Well in the standard domain
        ax.set_zlim(-2.0, 1.0)
        ax.set_zticks([-2.0, -1.0, 0.0, 1.0])

    # Internal zoom maintained from approved template
    ax.set_box_aspect(None, zoom=0.88)

    ax.view_init(elev=ELEV, azim=AZIM)
    ax.tick_params(colors="black", which="both")

    ax.legend(
        loc="upper right",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
    )

    if "_snap" in globals():
        _snap(ax)

# Manual margins and export
plt.subplots_adjust(left=0.01, right=0.97, top=0.88, bottom=0.02)

output_path = os.path.join(FIG_DIR, "fig04_mggp_surfaces.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.15)
plt.show()
print(f"Saved: {output_path}")

## Numerical Results

In [10]:
from symgene.metrics import rmse, nrmse

for pop_name, y_tr, y_v, y_te in [
        ("nguyen10",   y_n_train, y_n_val, y_n_test),
        ("doublewell", y_d_train, y_d_val, y_d_test)]:
    res   = results[pop_name]
    yp_tr = res.predict(X_train)
    yp_v  = res.predict(X_val)
    yp_te = res.predict(X_test)
    print(f"\n{'='*60}")
    print(f"Population : {pop_name.upper()}")
    print(f"Genes      : {res.n_genes_}")
    print(f"Expression : {res.best_expression_}")
    print(f"Train  RMSE={rmse(y_tr, yp_tr):.4f}  NRMSE={nrmse(y_tr, yp_tr):.4f}"
          f"  R²={r2_score(y_tr, yp_tr):.4f}")
    print(f"Val    RMSE={rmse(y_v,  yp_v ):.4f}  NRMSE={nrmse(y_v,  yp_v ):.4f}"
          f"  R²={r2_score(y_v,  yp_v ):.4f}")
    print(f"Test   RMSE={rmse(y_te, yp_te):.4f}  NRMSE={nrmse(y_te, yp_te):.4f}"
          f"  R²={r2_score(y_te, yp_te):.4f}")



Population : NGUYEN10
Genes      : 6
Expression : sin(x1) | cos(x2) | sin(x1) | cos(x2) | sin(x1) | cos(x2)
Train  RMSE=0.0000  NRMSE=0.0000  R²=1.0000
Val    RMSE=0.0000  NRMSE=0.0000  R²=1.0000
Test   RMSE=0.0000  NRMSE=0.0000  R²=1.0000

Population : DOUBLEWELL
Genes      : 10
Expression : square(x1) | square(x1) | square(square(x1)) | square(x2) | square(square(x2)) | square(x2) | square(x2) | square(x1) | square(x1) | square(x2)
Train  RMSE=0.0003  NRMSE=0.0001  R²=1.0000
Val    RMSE=0.0003  NRMSE=0.0001  R²=1.0000
Test   RMSE=0.0003  NRMSE=0.0001  R²=1.0000
